# SSL400 ST-GCN — Pure vs Augmented (restore ~64% accuracy)

**Previous good run:** ~121 classes (`min_samples ≥ 10`) → **val ~64.5% / test ~62.5%**

**If you saw ~48% val / ~33% F1:** Colab reused old all-class `data/processed` (or `min_samples=1`). This notebook **forces a rebuild** with the accuracy setup.

1. **Pure ST-GCN** — no aug
2. **Augmented ST-GCN** — mild aug + speed resample + focal γ=1

## Before you start

1. Upload latest `ssl400-research/` + `Dataset - MP - CSV/` to Drive
2. **Runtime → GPU (T4)**
3. Run cells **in order** (do not skip prepare)

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Go to project folder

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/SSL400/ssl400-research')
DATASET_DIR = Path('/content/drive/MyDrive/SSL400/Dataset - MP - CSV')

assert PROJECT_DIR.exists(), f'Missing project: {PROJECT_DIR}'
assert DATASET_DIR.exists(), f'Missing dataset: {DATASET_DIR}'

os.chdir(PROJECT_DIR)
print('CWD:', Path.cwd())

## 3) Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime to T4')
assert torch.cuda.is_available(), 'Enable GPU T4 before training'

## 4) Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Install done')

## 5) Write accuracy configs + clean old outputs

Rewrites `data.yaml`, `model_pure.yaml`, `model_augmented.yaml` so Drive cannot keep old all-class settings.

In [ ]:
from pathlib import Path

Path('configs').mkdir(exist_ok=True)

Path('configs/data.yaml').write_text(f'''
raw_csv_root: "{DATASET_DIR.as_posix()}"
processed_dir: "data/processed"
inspected_dir: "data/inspected"
min_samples_per_class: 10
sequence_length: 64
use_z: true
visibility_threshold: 0.5
mask_low_visibility: true
mask_lower_body: false
train_ratio: 0.70
val_ratio: 0.15
test_ratio: 0.15
split_seed: 42
save_tensors: true
'''.strip() + '\n', encoding='utf-8')

Path('configs/model_pure.yaml').write_text('''
model:
  in_channels: 3
  num_joints: 33
  num_classes: null
  channels: [64, 64, 128, 256]
  temporal_kernel: 9
  dropout: 0.5
  graph_strategy: spatial

train:
  epochs: 100
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 0.0001
  optimizer: adam
  scheduler: cosine
  early_stopping_patience: 20
  num_workers: 0
  amp: true
  class_weights: false
  weighted_sampler: false
  label_smoothing: 0.0
  focal_gamma: 0.0
  augment: false
  rare_class_augment: false
  seed: 42
  device: auto

paths:
  checkpoint_dir: "checkpoints"
  results_dir: "results"
  plots_dir: "results/plots"
  experiment_name: "stgcn_pure_seed42"
'''.strip() + '\n', encoding='utf-8')

Path('configs/model_augmented.yaml').write_text('''
model:
  in_channels: 3
  num_joints: 33
  num_classes: null
  channels: [64, 64, 128, 256]
  temporal_kernel: 9
  dropout: 0.5
  graph_strategy: spatial

train:
  epochs: 100
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 0.0001
  optimizer: adam
  scheduler: cosine
  early_stopping_patience: 20
  num_workers: 0
  amp: true
  class_weights: false
  weighted_sampler: false
  label_smoothing: 0.0
  focal_gamma: 1.0
  augment: true
  rare_class_augment: false
  seed: 42
  device: auto

paths:
  checkpoint_dir: "checkpoints"
  results_dir: "results"
  plots_dir: "results/plots"
  experiment_name: "stgcn_augmented_seed42"
'''.strip() + '\n', encoding='utf-8')

Path('configs/model.yaml').write_text(Path('configs/model_augmented.yaml').read_text(encoding='utf-8'), encoding='utf-8')

!python scripts/clean_for_retrain.py --also-processed --yes
print('Wrote accuracy configs + cleared results/checkpoints/processed')

## 6) Prepare dataset (MUST be ~121 classes)

If class count is ~300+, stop — that is the low-accuracy all-class setup.

In [ ]:
!python scripts/prepare_dataset.py --data-config configs/data.yaml

In [ ]:
import json
from pathlib import Path
import numpy as np

class_map = json.loads(Path('data/processed/class_to_idx.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('data/processed/manifest.json').read_text(encoding='utf-8'))
inv = json.loads(Path('data/inspected/dataset_inventory.json').read_text(encoding='utf-8'))
sample = manifest['train'][0]
tensor = np.load(sample['tensor_path'])

n_classes = len(class_map)
print('Classes:', n_classes)
print('min_samples_per_class:', inv.get('min_samples_per_class'))
print('Split sizes:', {k: len(v) for k, v in manifest.items()})
print('Example tensor:', sample['sample_id'], tensor.shape, tensor.dtype)

# Guard: previous good run used ~121 classes. All-class (~383) drops accuracy hard.
assert inv.get('min_samples_per_class') == 10, 'data.yaml must use min_samples_per_class=10'
assert 100 <= n_classes <= 150, (
    f'Expected ~121 classes for the 64% setup, got {n_classes}. '
    'Delete data/processed and re-run prepare with min_samples_per_class=10.'
)
print('OK: class filter matches the previous accuracy setup')

## 7A) Train 1 — Pure ST-GCN

Expect val acc climbing toward ~0.45–0.65 range (not stuck near 0.15).

In [ ]:
!python scripts/train_stgcn.py --data-config configs/data.yaml --model-config configs/model_pure.yaml

In [ ]:
!python scripts/evaluate_stgcn.py \
  --checkpoint checkpoints/stgcn_pure_seed42_best.pt \
  --split test \
  --data-config configs/data.yaml \
  --model-config configs/model_pure.yaml

## 7B) Train 2 — Augmented (previous ~62.5% test setup)

Target: **val ~64% / test ~62%** again.

In [ ]:
!python scripts/train_stgcn.py --data-config configs/data.yaml --model-config configs/model_augmented.yaml

In [ ]:
!python scripts/evaluate_stgcn.py \
  --checkpoint checkpoints/stgcn_augmented_seed42_best.pt \
  --split test \
  --data-config configs/data.yaml \
  --model-config configs/model_augmented.yaml

## 8) Comparison graphs

In [ ]:
!python scripts/compare_runs.py

## 9) MC Dropout + export (augmented / main model)

In [ ]:
!python scripts/mc_dropout_eval.py \
  --checkpoint checkpoints/stgcn_augmented_seed42_best.pt \
  --split val \
  --passes 20 \
  --data-config configs/data.yaml \
  --model-config configs/model_augmented.yaml

!python scripts/export_h5.py \
  --checkpoint checkpoints/stgcn_augmented_seed42_best.pt \
  --output checkpoints/stgcn_augmented_seed42_best.h5

## 10) Show graphs

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plots_dir = Path('results/plots')
wanted = [
    'stgcn_pure_seed42_loss_curves.png',
    'stgcn_pure_seed42_metric_curves.png',
    'stgcn_augmented_seed42_loss_curves.png',
    'stgcn_augmented_seed42_metric_curves.png',
    'compare_pure_vs_augmented_loss.png',
    'compare_pure_vs_augmented_val_metrics.png',
    'compare_pure_vs_augmented_overfit_gap.png',
    'compare_pure_vs_augmented_best_metrics.png',
]
for name in wanted:
    path = plots_dir / name
    if path.exists():
        print('Showing', path.name)
        display(Image(filename=str(path)))
    else:
        print('Missing:', name)

## 11) Summary — check you recovered previous accuracy

In [ ]:
import json
from pathlib import Path

PREV_VAL_ACC = 0.645
PREV_TEST_ACC = 0.625

for exp in ('stgcn_pure_seed42', 'stgcn_augmented_seed42'):
    summary_path = Path(f'results/{exp}_train_summary.json')
    metrics_path = Path(f'results/{exp}_test_metrics.json')
    print('=' * 60)
    print(exp)
    if summary_path.exists():
        s = json.loads(summary_path.read_text())
        print('classes', s.get('num_classes'), 'best_epoch', s.get('best_epoch'))
        print('val_acc', s.get('best_val_accuracy'), 'val_f1', s.get('best_val_f1_macro'))
        print('mode', s.get('training_mode'))
    if metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        print('test_acc', m.get('accuracy'), 'test_f1', m.get('f1_macro'), 'top5', m.get('top5_accuracy'))

aug_sum = Path('results/stgcn_augmented_seed42_train_summary.json')
aug_test = Path('results/stgcn_augmented_seed42_test_metrics.json')
if aug_sum.exists():
    s = json.loads(aug_sum.read_text())
    print('=' * 60)
    print(f"Target recovery: val~{PREV_VAL_ACC:.1%}, test~{PREV_TEST_ACC:.1%}")
    print(f"This run val_acc={s.get('best_val_accuracy'):.4f} (classes={s.get('num_classes')})")
    if s.get('num_classes', 999) > 200:
        print('WARNING: too many classes — rebuild with min_samples_per_class=10')
    elif s.get('best_val_accuracy', 0) < 0.55:
        print('WARNING: still far below previous accuracy — check GPU, class count, and configs')
    else:
        print('Looking good vs previous accuracy band')
if aug_test.exists():
    m = json.loads(aug_test.read_text())
    print(f"This run test_acc={m.get('accuracy'):.4f}")

cmp_path = Path('results/compare_pure_vs_augmented_summary.json')
if cmp_path.exists():
    print('Comparison:', json.dumps(json.loads(cmp_path.read_text()), indent=2))